# E-commerce Funnel Analysis (October 2019)

**Business question:** where do shoppers drop off before buying, and which product category is worth investigating first?

**Data:** Kaggle, *E-commerce behavior data from multi-category store* (October 2019 file, about 42M events, one row = one user action on one product at one moment).

**How to read this notebook:** each section states the question, the result, what it means, and the limitation. Numbers in the text are from the October 2019 run. Everything is observational: nothing here proves a cause.

**To run it:** set `DATA_PATH` and `OUTPUT_DIR` in the first code cell. Needs `pandas`, `numpy`, `statsmodels`. The full run needs a lot of memory (it was developed on Kaggle).

## 1. Setup and load

**Question:** what is in the data, and what is one row?

**Why these columns:** `user_id`, `event_type`, `event_time` define the funnel. `product_id` and `category_code` allow category work. `price` is only used for the revenue scenario. `user_session` is not used, so it is not loaded (saves memory).

**Note:** times are loaded as timezone-naive. All data is UTC, so every cutoff below is also naive.

In [1]:
import pandas as pd
import numpy as np

# --- Change these paths if you run outside Kaggle ---
DATA_PATH = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv'
OUTPUT_DIR = '/kaggle/working/'

dtypes = {
    'event_type': 'category',
    'product_id': 'int32',
    'category_id': 'int64',
    'category_code': 'category',
    'brand': 'category',
    'price': 'float32',
    'user_id': 'int64',
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    usecols=['event_time', 'event_type', 'product_id', 'category_id',
             'category_code', 'brand', 'price', 'user_id'],
)
# Parse with an explicit format (much faster than letting pandas guess)
df['event_time'] = pd.to_datetime(df['event_time'], format='%Y-%m-%d %H:%M:%S UTC')

print(df.shape)   # expect (42448764, 8) before removing duplicates
print(df.dtypes)
df.head()

(42448764, 8)
event_time       datetime64[ns]
event_type             category
product_id                int32
category_id               int64
category_code          category
brand                  category
price                   float32
user_id                   int64
dtype: object


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.790001,541312140
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.200001,554748717
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.099976,519107250
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217


## 2. Data quality

Each check asks what would break if we skipped it.

**Result summary**
- `category_code` is missing in **31.84%** of rows and `brand` in **14.41%**. Missingness is concentrated by brand (some brands 100% missing, others 0%), so it is not random.
- **35,207** duplicate events (**0.083%**), spread thinly across users (top 10 users are about 2.5% of them), so they look like logging noise. They are dropped.
- Every product is either fully labeled or fully unlabeled (**73,150** labeled, **93,644** unlabeled, **0** mixed). Category results therefore cover labeled products only.
- Only **259** user-product pairs have a purchase with no view or cart for that product (0.05%). **13,676** have a cart recorded after the purchase.

**Limitation:** category results exclude unlabeled products, so they may not describe the whole store.

### 2a. Missing values

In [2]:
(df.isna().sum() / len(df) * 100).round(2)

event_time        0.00
event_type        0.00
product_id        0.00
category_id       0.00
category_code    31.84
brand            14.41
price             0.00
user_id           0.00
dtype: float64

In [3]:
# Is category_code missingness random or concentrated? Check by brand.
missing_by_brand = df.groupby('brand', observed=True)['category_code'].apply(
    lambda x: x.isna().mean() * 100
).sort_values(ascending=False)

print(missing_by_brand.head(10))
print(missing_by_brand.tail(10))

brand
skindoctors           100.0
a-case                100.0
a-derma               100.0
velvet                100.0
salvatoreferragamo    100.0
novatex               100.0
bsa-wheels            100.0
dormeo                100.0
tundracomfort         100.0
spinmop               100.0
Name: category_code, dtype: float64
brand
emer           0.0
nodor          0.0
nowodvorski    0.0
nuna           0.0
shindo         0.0
laudio         0.0
laurastar      0.0
leadbros       0.0
leader         0.0
neumann        0.0
Name: category_code, dtype: float64


### 2b. Duplicates
A duplicate event is the same user, product, time and event type. Check size first, then whether duplicates are concentrated in a few users.

In [4]:
dup_cols = ['user_id', 'product_id', 'event_time', 'event_type']
dup_flag = df.duplicated(subset=dup_cols)
n_dup = int(dup_flag.sum())
print(f"Duplicate events: {n_dup:,} ({n_dup / len(df) * 100:.4f}% of rows)")

# Duplicates per user, as a share of all duplicates
dup_by_user = dup_flag.groupby(df['user_id']).sum().sort_values(ascending=False)
print("Top 10 users, share of all duplicates (%):")
print((dup_by_user / n_dup * 100).round(2).head(10))
print(f"Top 10 users combined: {dup_by_user.head(10).sum() / n_dup * 100:.2f}%")

Duplicate events: 35,207 (0.0829% of rows)
Top 10 users, share of all duplicates (%):
user_id
538557901    0.53
517728689    0.39
548471261    0.27
563571758    0.22
547516438    0.22
513032906    0.20
555130288    0.18
552601624    0.18
520248949    0.16
560549276    0.16
dtype: float64
Top 10 users combined: 2.53%


In [5]:
# Drop duplicates (done once, here, after the checks)
df = df.drop_duplicates(subset=dup_cols)
del dup_flag, dup_by_user
print(df.shape)   # expect (42413557, 8)

(42413557, 8)


### 2c. Do missing labels belong to the product?

In [6]:
share_labeled = df.groupby('product_id')['category_code'].apply(lambda x: x.notna().mean())

print("Fully labeled products:  ", (share_labeled == 1).sum())
print("Fully unlabeled products:", (share_labeled == 0).sum())
print("Mixed products:          ", ((share_labeled > 0) & (share_labeled < 1)).sum())

Fully labeled products:   73150
Fully unlabeled products: 93644
Mixed products:           0


### 2d. Impossible or unusual sequences
Earliest time of each event type for each user-product pair. A purchase with no view or cart is a legitimate edge case (direct link, or activity before October), so it is flagged, not deleted.

In [7]:
event_times = df.groupby(['user_id', 'product_id', 'event_type'], observed=True)['event_time'].min().unstack()

no_prior_activity = (event_times['purchase'].notna()
                     & event_times['view'].isna()
                     & event_times['cart'].isna())
print(f"Purchases with NO prior view/cart at all: {no_prior_activity.sum()}")
print(f"Percentage of all user-product purchase pairs: "
      f"{no_prior_activity.sum() / event_times['purchase'].notna().sum() * 100:.2f}%")

cart_after_purchase = (event_times['cart'].notna()
                       & event_times['purchase'].notna()
                       & (event_times['cart'] > event_times['purchase']))
print(f"Cart events recorded after purchase: {cart_after_purchase.sum()}")

Purchases with NO prior view/cart at all: 259
Percentage of all user-product purchase pairs: 0.05%
Cart events recorded after purchase: 13676


## 3. How long between cart and purchase? (choosing a window)

**Question:** how far back should a cart count as connected to a purchase?

**Result:** median gap under 2 minutes, 90th percentile about 22 minutes, 95th about 17 hours, 99th about 7 days. The distribution is right-skewed (the mean is dragged up by a few slow buyers), so we use percentiles, not the mean. 

**Decision: a 24-hour window.** It covers the 95th percentile without linking carts and purchases that are probably unrelated.

**Limitation:** gaps are measured from the *first* cart per user-product pair, so true gaps from the nearest cart are shorter. 24 hours is therefore conservative.

In [8]:
cart_before_purchase = (event_times['cart'].notna()
                        & event_times['purchase'].notna()
                        & (event_times['cart'] < event_times['purchase']))
print(f"User-product pairs with a cart BEFORE the purchase: {cart_before_purchase.sum():,}")

time_to_purchase = (event_times.loc[cart_before_purchase, 'purchase']
                    - event_times.loc[cart_before_purchase, 'cart'])
print(time_to_purchase.describe())
print(time_to_purchase.quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

# Free memory: event_times is large and not needed again
del event_times, cart_before_purchase, cart_after_purchase, no_prior_activity, time_to_purchase

User-product pairs with a cart BEFORE the purchase: 246,010
count                       246010
mean     0 days 05:53:52.762338929
std      1 days 11:59:01.104402257
min                0 days 00:00:06
25%                0 days 00:00:51
50%                0 days 00:01:50
75%                0 days 00:04:03
max               30 days 07:29:47
dtype: object
0.50             0 days 00:01:50
0.75             0 days 00:04:03
0.90             0 days 00:22:03
0.95   0 days 16:46:50.499999999
0.99      7 days 16:38:28.870000
dtype: timedelta64[ns]


## 4. The funnel at user level

**Question:** where are users lost, and does a simple view, cart, purchase line describe how people buy?

**Unit:** unique users, not rows. Repeat views by one user would otherwise inflate every stage.

**Result:** 3,022,130 users viewed. Of **347,056** who viewed and bought, **202,754 (58.4%)** used the cart and **144,302 (41.6%)** bought without carting. So low view-to-cart does **not** mean most viewers are lost at the cart: many buy without it. The whole-month any-route rate is 11.48%.

**Limitation:** whole-month user counts ignore event order, product matching and the time window. Use the 7-day cohort below for a fairer rate.

In [9]:
viewers_all    = set(df.loc[df['event_type'] == 'view', 'user_id'].unique())
carters_all    = set(df.loc[df['event_type'] == 'cart', 'user_id'].unique())
purchasers_all = set(df.loc[df['event_type'] == 'purchase', 'user_id'].unique())

# Every purchaser falls into exactly one of four buckets
b1 = purchasers_all & viewers_all & carters_all          # view + cart + purchase
b2 = (purchasers_all & viewers_all) - carters_all        # view -> purchase, skipped cart
b3 = (purchasers_all & carters_all) - viewers_all        # cart -> purchase, no view recorded
b4 = purchasers_all - viewers_all - carters_all          # purchase only

print(f"Viewers:                                {len(viewers_all):,}")
print(f"View + Cart + Purchase (used the cart): {len(b1):,}")
print(f"View -> Purchase (skipped the cart):    {len(b2):,}")
print(f"Cart -> Purchase (no view recorded):    {len(b3):,}")
print(f"Purchase only (no view, no cart):       {len(b4):,}")
print(f"Total purchasers:                       {len(purchasers_all):,}")

# Buyers who also viewed, by route (the base for the cart / no-cart split)
buyers_any_route = len(b1) + len(b2)
print(f"\nViewers who bought by any route: {buyers_any_route:,} "
      f"({buyers_any_route / len(viewers_all) * 100:.2f}% of viewers, whole month)")
print(f"Share who used the cart:  {len(b1) / buyers_any_route * 100:.1f}%")
print(f"Share who skipped it:     {len(b2) / buyers_any_route * 100:.1f}%")

Viewers:                                3,022,130
View + Cart + Purchase (used the cart): 202,754
View -> Purchase (skipped the cart):    144,302
Cart -> Purchase (no view recorded):    23
Purchase only (no view, no cart):       39
Total purchasers:                       347,118

Viewers who bought by any route: 347,056 (11.48% of viewers, whole month)
Share who used the cart:  58.4%
Share who skipped it:     41.6%


## 5. A fair rate: the 7-day cohort

**Problem:** a user who first viewed on Oct 29 has only 2 days left in the data, so counting them as a non-buyer is unfair (right-censoring).

**Fix:** include only users whose **first view is before Oct 25**, and count a purchase only if it happens **within 7 days of that first view**. The cutoff applies to the first view, not to the purchase.

**Choosing 7 days:** among buyers, the median first-view-to-purchase gap is about 2.5 hours, but 25% take more than about 6 days. 7 days keeps most of October in the sample; 20 days would keep only about a third of it.

**Result:** **9.21%** of the cohort (235,860 of 2,559,699) bought within 7 days.

**Limitation:** this is a **lower bound** on true conversion. Buyers who take longer than 7 days are counted as non-buyers. It covers October only and users who first viewed before Oct 25.

In [10]:
# Gap from a user's first view to first purchase (used to choose the window)
first_view = df[df['event_type'] == 'view'].groupby('user_id')['event_time'].min()
first_purchase_any = df[df['event_type'] == 'purchase'].groupby('user_id')['event_time'].min()

both = pd.concat([first_view.rename('first_view'),
                  first_purchase_any.rename('first_purchase')], axis=1).dropna()
gap_view_to_purchase = both['first_purchase'] - both['first_view']
gap_view_to_purchase = gap_view_to_purchase[gap_view_to_purchase >= pd.Timedelta(0)]

print(f"Users with both a view and a purchase: {len(both):,}")
print(f"Users with first view before/at first purchase: {len(gap_view_to_purchase):,}")
print(gap_view_to_purchase.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

Users with both a view and a purchase: 347,056
Users with first view before/at first purchase: 346,756
0.25           0 days 00:04:34
0.50    0 days 02:25:55.500000
0.75           6 days 07:07:43
0.90          14 days 18:54:06
0.95   19 days 16:24:22.750000
0.99   25 days 23:26:21.250000
dtype: timedelta64[ns]


In [11]:
print("Data starts:", df['event_time'].min())
print("Data ends:  ", df['event_time'].max())

# Cohort: users whose first view is before Oct 25 (so they have a full 7 days left)
cutoff = pd.Timestamp('2019-10-25')
cohort = first_view[first_view < cutoff]

purchases = df.loc[df['event_type'] == 'purchase', ['user_id', 'event_time']]
purchases = purchases[purchases['user_id'].isin(cohort.index)]
purchases = purchases.join(cohort.rename('first_view'), on='user_id')

# Keep purchases within 0 to 7 days AFTER the first view
gap = purchases['event_time'] - purchases['first_view']
valid = purchases[(gap >= pd.Timedelta(0)) & (gap <= pd.Timedelta(days=7))]

n_cohort = int(len(cohort))
n_converted = int(valid['user_id'].nunique())   # unique users, not purchase rows
rate_7d = n_converted / n_cohort

print(f"Cohort size (first view before Oct 25): {n_cohort:,}")
print(f"Converted within 7 days of first view:  {n_converted:,}")
print(f"7-day conversion rate:                  {rate_7d * 100:.2f}%")

Data starts: 2019-10-01 00:00:00
Data ends:   2019-10-31 23:59:59
Cohort size (first view before Oct 25): 2,559,699
Converted within 7 days of first view:  235,860
7-day conversion rate:                  9.21%


## 6. Does conversion differ by category?

**Question:** does the 9.21% hide big differences between categories? The decision it supports: which categories to investigate first.

**Rule:** a user converts in a category only if they bought a product *in that category* within 7 days of their first view *in that category*. Users can appear in several categories, so category counts overlap and do not add up to the cohort.

**Scope:** labeled products only. Sub-category results are reported only for categories with at least **3,000 viewers**, because one buyer more or less barely moves the rate of a large group but swings a small one.

**Result:** 125 sub-categories, 93 kept. Smartphones convert at **11.04%** (1,073,622 viewers). Apparel sub-categories are near zero (for example dresses 0.04%). At top level, apparel is **1.74%** and furniture **2.54%**, against a 9.21% baseline.

**Limitation:** the strict matching rule explains only part of apparel's low rate (it rises from about 0.5% to 1.74% when widened to top level). Low demand, fit or price are possible reasons, but this data cannot tell us why.

### 6a. Group sizes (choosing the minimum)

In [12]:
cat_views = df[(df['event_type'] == 'view') & df['category_code'].notna()]
cat_views = cat_views[cat_views['event_time'] < cutoff]

first_view_cat = cat_views.groupby(['user_id', 'category_code'], observed=True)['event_time'].min()
viewers_per_cat = (first_view_cat.reset_index()
                   .groupby('category_code', observed=True)['user_id'].nunique()
                   .sort_values(ascending=False))

print(f"Number of categories: {len(viewers_per_cat)}")
print(viewers_per_cat.describe())
print(viewers_per_cat.head(10))

del cat_views, first_view_cat

Number of categories: 125
count    1.250000e+02
mean     2.896258e+04
std      9.954397e+04
min      1.650000e+02
25%      2.859000e+03
50%      9.190000e+03
75%      2.343500e+04
max      1.073622e+06
Name: user_id, dtype: float64
category_code
electronics.smartphone              1073622
electronics.audio.headphone          176310
electronics.clocks                   175830
electronics.video.tv                 135496
appliances.kitchen.washer            107753
appliances.kitchen.refrigerators     107701
apparel.shoes                        106305
computers.notebook                   105627
appliances.environment.vacuum         97604
apparel.shoes.keds                    77039
Name: user_id, dtype: int64


### 6b. Conversion by sub-category

In [13]:
labeled = df[df['category_code'].notna()]

# 1. Each user's first view per category (before the cutoff)
v = labeled[(labeled['event_type'] == 'view') & (labeled['event_time'] < cutoff)]
fv = v.groupby(['user_id', 'category_code'], observed=True)['event_time'].min().reset_index()
fv = fv.rename(columns={'event_time': 'first_view'})

# 2. Purchases in labeled categories
p = labeled[labeled['event_type'] == 'purchase'][['user_id', 'category_code', 'event_time']]
p = p.rename(columns={'event_time': 'purchase_time'})

# 3. Match purchases to first views (same user AND same category)
m = fv.merge(p, on=['user_id', 'category_code'], how='left')
gap = m['purchase_time'] - m['first_view']
m['converted'] = (gap >= pd.Timedelta(0)) & (gap <= pd.Timedelta(days=7))

# 4. One row per user-category: did they convert at all?
uc = m.groupby(['user_id', 'category_code'], observed=True)['converted'].max().reset_index()

# 5. Category summary, with the minimum-size rule
res_sub = uc.groupby('category_code', observed=True).agg(
    viewers=('user_id', 'nunique'),
    converters=('converted', 'sum'),
)
res_sub['conv_rate_%'] = (res_sub['converters'] / res_sub['viewers'] * 100).round(2)
res_sub = res_sub[res_sub['viewers'] >= 3000].sort_values('conv_rate_%')

print(f"Categories kept: {len(res_sub)}")
print(res_sub.head(10))   # lowest conversion
print(res_sub.tail(10))   # highest conversion

del v, fv, p, m, uc, gap

Categories kept: 93
                             viewers  converters  conv_rate_%
category_code                                                
apparel.dress                  13666           6         0.04
auto.accessories.winch          3491           7         0.20
apparel.trousers                8690          18         0.21
apparel.tshirt                  9826          33         0.34
apparel.shirt                   6478          22         0.34
apparel.jeans                  24681          98         0.40
apparel.underwear              13810          59         0.43
apparel.costume                44121         305         0.69
auto.accessories.parktronic     4499          33         0.73
apparel.shoes.moccasins         5231          40         0.76
                                     viewers  converters  conv_rate_%
category_code                                                        
appliances.environment.vacuum          97604        6880         7.05
appliances.kitchen.washer 

### 6c. Conversion by top-level category
Widening the rule to the top level (the text before the first dot, for example `apparel`) tests whether the strict matching rule was hiding buyers.

In [14]:
labeled = labeled.copy()
labeled['top_cat'] = labeled['category_code'].astype(str).str.split('.').str[0]

v = labeled[(labeled['event_type'] == 'view') & (labeled['event_time'] < cutoff)]
fv = v.groupby(['user_id', 'top_cat'])['event_time'].min().reset_index()
fv = fv.rename(columns={'event_time': 'first_view'})

p = labeled[labeled['event_type'] == 'purchase'][['user_id', 'top_cat', 'event_time']]
p = p.rename(columns={'event_time': 'purchase_time'})

m = fv.merge(p, on=['user_id', 'top_cat'], how='left')
gap = m['purchase_time'] - m['first_view']
m['converted'] = (gap >= pd.Timedelta(0)) & (gap <= pd.Timedelta(days=7))

uc = m.groupby(['user_id', 'top_cat'])['converted'].max().reset_index()
res_top = uc.groupby('top_cat').agg(viewers=('user_id', 'nunique'),
                                    converters=('converted', 'sum'))
res_top['conv_rate_%'] = (res_top['converters'] / res_top['viewers'] * 100).round(2)
print(res_top.sort_values('conv_rate_%'))

del labeled, v, fv, p, m, uc, gap

              viewers  converters  conv_rate_%
top_cat                                       
country_yard     3361          48         1.43
apparel        216093        3769         1.74
sport           33275         627         1.88
accessories     41837         847         2.02
furniture      134881        3427         2.54
kids            79871        2715         3.40
stationery       1692          60         3.55
auto           137934        5718         4.15
construction    82727        3979         4.81
computers      199885       12010         6.01
medicine         2698         198         7.34
appliances     450824       35909         7.97
electronics   1404110      144934        10.32


## 7. Do buyers browse more than non-buyers?

**Question:** would "show more similar products" help? Measure: distinct products viewed within 24 hours of first view. For buyers the window stops at the first purchase, so views after buying do not count (they may be caused by buying).

**Result:** the medians are identical (**2** for both groups). Buyers are only modestly higher in the upper tail (95th percentile 15 vs 12). Most people, buyers or not, look at very few products.

**Limitation:** buyers who purchase quickly get a shorter window than non-buyers, so the comparison is slightly tilted against buyers. Even if buyers browsed more, that would not show that browsing causes buying.

In [15]:
# 1. Cohort: each user's first view, before the cutoff
views = df.loc[df['event_type'] == 'view', ['user_id', 'product_id', 'event_time']]
first_view = views.groupby('user_id')['event_time'].min()
cohort = first_view[first_view < cutoff].rename('first_view')

# 2. Each buyer's first purchase within 7 days AFTER their first view
purch = df.loc[df['event_type'] == 'purchase', ['user_id', 'event_time']]
purch = purch[purch['user_id'].isin(cohort.index)].join(cohort, on='user_id')
gap = purch['event_time'] - purch['first_view']
purch = purch[(gap >= pd.Timedelta(0)) & (gap <= pd.Timedelta(days=7))]
first_purchase = purch.groupby('user_id')['event_time'].min().rename('first_purchase')

# 3. One row per cohort user: buyer or not, and where their window ends
info = cohort.to_frame().join(first_purchase)
info['is_buyer'] = info['first_purchase'].notna()
info['window_end'] = info['first_view'] + pd.Timedelta(hours=24)
stop_early = info['is_buyer'] & (info['first_purchase'] < info['window_end'])
info.loc[stop_early, 'window_end'] = info.loc[stop_early, 'first_purchase']

# 4. Distinct products viewed inside each user's window
v = views[views['user_id'].isin(cohort.index)].join(
    info[['first_view', 'window_end']], on='user_id')
v = v[(v['event_time'] >= v['first_view']) & (v['event_time'] <= v['window_end'])]
info['products_viewed'] = v.groupby('user_id')['product_id'].nunique()

# 5. Compare buyers vs non-buyers (median and percentiles, not the mean)
print(info['is_buyer'].value_counts())
print(info.groupby('is_buyer')['products_viewed']
          .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

del views, v, purch, gap, info

is_buyer
False    2323839
True      235860
Name: count, dtype: int64
              count      mean       std  min  25%  50%  75%   90%   95%  \
is_buyer                                                                  
False     2323839.0  3.507496  5.168440  1.0  1.0  2.0  4.0   8.0  12.0   
True       235860.0  4.227330  6.364806  1.0  1.0  2.0  5.0  10.0  15.0   

             max  
is_buyer          
False     1015.0  
True       235.0  


## 8. Is a gap real, or luck? (plausible ranges)

**Question:** a rate is a best guess. How far could luck move it? A 95% plausible range (Wilson confidence interval) shows this. Overlapping ranges mean we cannot tell groups apart; separated ranges mean the gap is very unlikely to be luck.

**Result:** the overall 7-day rate is **9.21%** (range 9.18% to 9.25%). Apparel **1.74%** (1.69% to 1.80%) and furniture **2.54%** (2.46% to 2.63%) do not overlap, so that gap is very likely real.

**What this does not say:** a narrow range means the number is **precise**, not **correct**. The 7-day window, the cohort rule and one month of data still limit what it means. Also, "real" is not the same as "important": whether to act depends on order value and the cost of a fix. Users overlap between categories, so treat this as a guide.

In [16]:
from statsmodels.stats.proportion import proportion_confint

low, high = proportion_confint(n_converted, n_cohort, alpha=0.05, method='wilson')
print(f"Overall 7-day rate: {n_converted / n_cohort * 100:.2f}%   range {low * 100:.2f}% to {high * 100:.2f}%")

for name in ['apparel', 'furniture']:
    buyers = int(res_top.loc[name, 'converters'])
    viewers = int(res_top.loc[name, 'viewers'])
    low, high = proportion_confint(buyers, viewers, alpha=0.05, method='wilson')
    print(f"{name:10s} rate {buyers / viewers * 100:.2f}%   range {low * 100:.2f}% to {high * 100:.2f}%")

Overall 7-day rate: 9.21%   range 9.18% to 9.25%
apparel    rate 1.74%   range 1.69% to 1.80%
furniture  rate 2.54%   range 2.46% to 2.63%


## 9. Category opportunity (a scenario, not a forecast)

**Question:** which category should be investigated first? A low rate alone is not enough, because volume matters: 1 point of conversion on 200,000 viewers is far more than on 500.

**Metric:** extra buyers *if* a category matched the overall 7-day rate = viewers x (baseline - category rate). Above-baseline categories get 0.

**Result:** apparel is the biggest opportunity (about **16,133** extra buyers), then furniture (about 8,996) and auto (about 6,986).

**Limitation:** the baseline is an **assumption**. Apparel may never convert like electronics, so these are scenarios. Users overlap across categories, so do not add the rows together. Labeled products only.

In [17]:
benchmark = round(n_converted / n_cohort, 4)   # ASSUMPTION: overall 7-day rate, rounded to 9.21%

opp = res_top.rename(columns={'conv_rate_%': 'rate_%'})
opp = opp[opp['viewers'] >= 3000].copy()       # same minimum-size rule as before

shortfall = (benchmark - opp['converters'] / opp['viewers']).clip(lower=0)
opp['extra_buyers_if_baseline'] = (opp['viewers'] * shortfall).round(0).astype(int)
opp = opp.sort_values('extra_buyers_if_baseline', ascending=False)

print(f"Benchmark used: {benchmark * 100:.2f}%")
print(opp)

Benchmark used: 9.21%
              viewers  converters  rate_%  extra_buyers_if_baseline
top_cat                                                            
apparel        216093        3769    1.74                     16133
furniture      134881        3427    2.54                      8996
auto           137934        5718    4.15                      6986
computers      199885       12010    6.01                      6399
appliances     450824       35909    7.97                      5612
kids            79871        2715    3.40                      4641
construction    82727        3979    4.81                      3640
accessories     41837         847    2.02                      3006
sport           33275         627    1.88                      2438
country_yard     3361          48    1.43                       262
electronics   1404110      144934   10.32                         0


## 10. Experiment design (planned, NOT run)

**Why an experiment:** the data shows apparel converts low but not *why*. Comparing people who chose to read a size guide with people who did not would be unfair (keen buyers read more). Random assignment makes the two groups alike in everything except the change.

**Hypothesis:** adding a size guide to apparel product pages will increase the share of apparel viewers who buy within 7 days.

**Primary metric:** share who buy within 7 days (baseline about 1.74%). **Guardrail:** share of orders returned (not in this data).

**Status:** this is a design. October's data is a record of what already happened, so there is **no result**. Nobody was randomly assigned and nothing on the live site changed.

**Sample size:** depends on the smallest lift worth detecting. Smaller lifts need many more users.

In [18]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

baseline = round(res_top.loc['apparel', 'converters'] / res_top.loc['apparel', 'viewers'], 4)
lifts_pts = [0.1, 0.25, 0.5]   # absolute lift in percentage points
sample_sizes = {}

for lift in lifts_pts:
    target = baseline + lift / 100
    effect = proportion_effectsize(target, baseline)
    n = NormalIndPower().solve_power(effect_size=effect, alpha=0.05,
                                     power=0.8, alternative='two-sided')
    sample_sizes[lift] = int(n)
    print(f"Lift of {lift:>4} points ({baseline * 100:.2f}% -> {target * 100:.2f}%): "
          f"about {int(n):,} users per group")

Lift of  0.1 points (1.74% -> 1.84%): about 275,906 users per group
Lift of 0.25 points (1.74% -> 1.99%): about 45,917 users per group
Lift of  0.5 points (1.74% -> 2.24%): about 12,198 users per group


## 11. Business impact scenario

Each number is labeled **OBSERVED** (measured), **ASSUMPTION** (chosen), or **SCENARIO** (what would happen if the assumption were true).

**Limitations:** there is no order ID, so revenue is per purchased **item**. `price` is revenue, not profit (no cost or margin data), and its currency is not stated. The extra-buyers figure only exists if a size guide really adds 0.25 points, which is unknown.

In [19]:
# Sanity check on price across ALL purchase events
purch = df[df['event_type'] == 'purchase']
print(purch['price'].describe())
print((purch['price'] <= 0).sum(), "purchase events with price <= 0")

# Apparel purchases (category_code starts with 'apparel')
is_apparel = purch['category_code'].astype('string').str.startswith('apparel', na=False)
apparel = purch[is_apparel]
print(len(apparel), "apparel purchase events (items, not buyers)")
print(apparel['price'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

count    742773.000000
mean        309.560577
std         348.951050
min           0.770000
25%          87.489998
50%         179.839996
75%         375.209991
max        2574.070068
Name: price, dtype: float64
0 purchase events with price <= 0
8002 apparel purchase events (items, not buyers)
count    8002.000000
mean       78.097694
std        36.697964
min         4.380000
25%        51.220001
50%        71.820000
75%       100.389999
90%       127.419998
95%       137.710007
max       324.329987
Name: price, dtype: float64


In [20]:
apparel_viewers = int(res_top.loc['apparel', 'viewers'])        # OBSERVED
median_price = float(apparel['price'].median())                 # OBSERVED (per item)
assumed_lift = 0.0025                                           # ASSUMPTION: +0.25 points

extra_buyers = apparel_viewers * assumed_lift                   # SCENARIO
extra_revenue = extra_buyers * median_price                     # SCENARIO (one item per extra buyer)

print(f"Apparel viewers (observed):              {apparel_viewers:,}")
print(f"Median apparel item price (observed):    {median_price:.2f}")
print(f"Assumed lift:                            {assumed_lift * 100:.2f} points")
print(f"Extra buyers (scenario):                 {extra_buyers:,.0f}")
print(f"Extra revenue per month (scenario):      {extra_revenue:,.0f}   (before costs)")

Apparel viewers (observed):              216,093
Median apparel item price (observed):    71.82
Assumed lift:                            0.25 points
Extra buyers (scenario):                 540
Extra revenue per month (scenario):      38,799   (before costs)


## 12. Export small summary tables for the dashboard

The dashboard (Tableau Public) reads three small CSVs, not the 42M-row data. Each table says what basis its numbers use.

In [22]:
funnel = pd.DataFrame({
    'metric': ['Cohort users (first view before Oct 25)',
               'Bought within 7 days of first view',
               '7-day conversion rate %',
               'Buyers who used the cart (all of October)',
               'Buyers who skipped the cart (all of October)'],
    'value': [n_cohort, n_converted, round(n_converted / n_cohort * 100, 2), len(b1), len(b2)],
    'basis': ['7-day cohort', '7-day cohort', '7-day cohort',
              'All October, no 7-day rule', 'All October, no 7-day rule'],
})

opportunity = opp.reset_index().rename(columns={'top_cat': 'category'})
opportunity['note'] = 'Scenario, labeled products only, categories overlap'

test_plan = pd.DataFrame({
    'item': ['Hypothesis', 'Primary metric', 'Guardrail', 'Smallest lift to detect',
             'Users per group', 'Status'],
    'detail': ['Size guide on apparel pages raises 7-day purchase share',
               f'Share of apparel viewers who buy within 7 days (baseline {baseline * 100:.2f}%)',
               'Share of orders returned (not in our data)',
               f'0.25 points ({baseline * 100:.2f}% to {(baseline + 0.0025) * 100:.2f}%)',
               f'about {sample_sizes[0.25]:,}',
               'Planned, NOT run. No result.'],
})

funnel.to_csv(OUTPUT_DIR + 'funnel.csv', index=False)
opportunity.to_csv(OUTPUT_DIR + 'opportunity.csv', index=False)
test_plan.to_csv(OUTPUT_DIR + 'test_plan.csv', index=False)
print(opportunity[['category', 'viewers', 'rate_%', 'extra_buyers_if_baseline']].head(3))

    category  viewers  rate_%  extra_buyers_if_baseline
0    apparel   216093    1.74                     16133
1  furniture   134881    2.54                      8996
2       auto   137934    4.15                      6986


## Limitations

- One month of data (October). No retention or seasonality can be measured.
- Category results cover labeled products only (about 44% of products); the rest have no `category_code`.
- The 7-day rate is a lower bound: slow buyers and users who first viewed after Oct 24 are not counted.
- `price` is revenue, not profit. No cost, margin, returns or traffic-source data.
- No order ID, so revenue is estimated per purchased item.
- Observational data only. Nothing here proves a cause. The proposed size-guide test is a plan, not a result.

## What I would ask the company for

Returns data (the guardrail), cost and margin, traffic source, category labels for the unlabeled products, and more than one month of history.